# Resume Full Run from epoch 26
**Resumes from last.pt saved in Drive**

1. Runtime → Change runtime type → T4 GPU
2. Run all

In [ ]:
# Mount Drive (new account) and download last.pt from shared link
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, shutil

# Paste the shared link to last.pt from your old Drive account
SHARED_LAST_PT = 'PASTE_SHARED_LINK_TO_last.pt_HERE'

# Extract file ID
if 'id=' in SHARED_LAST_PT:
    file_id = SHARED_LAST_PT.split('id=')[1].split('&')[0]
elif '/d/' in SHARED_LAST_PT:
    file_id = SHARED_LAST_PT.split('/d/')[1].split('/')[0]
else:
    raise ValueError('Could not extract file ID from link')

subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
import gdown
last_pt = '/content/last.pt'
gdown.download(id=file_id, output=last_pt, quiet=False)
print(f'Downloaded last.pt: {os.path.getsize(last_pt)/1e6:.1f} MB')

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())


In [ ]:
# Install ultralytics
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics>=8.4.0'], check=True)
import ultralytics
print(f'ultralytics {ultralytics.__version__}')


In [ ]:
# Download dataset from shared Drive link (old account)
import os, subprocess, tarfile, yaml

# Paste shared link to backup_merged_dataset.tar.gz from old account
SHARED_TAR = 'PASTE_SHARED_LINK_TO_backup_merged_dataset.tar.gz_HERE'

if 'id=' in SHARED_TAR:
    tar_id = SHARED_TAR.split('id=')[1].split('&')[0]
elif '/d/' in SHARED_TAR:
    tar_id = SHARED_TAR.split('/d/')[1].split('/')[0]
else:
    raise ValueError('Could not extract file ID')

import gdown
tar_path = '/content/backup_merged_dataset.tar.gz'
gdown.download(id=tar_id, output=tar_path, quiet=False)
print(f'Downloaded: {os.path.getsize(tar_path)/1e9:.2f} GB')

print('Extracting...')
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

data_yaml = os.path.join(extract_dir, 'merged_dataset', 'data.yaml')
base = os.path.join(extract_dir, 'merged_dataset')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)

for split in ['train', 'val', 'test']:
    print(f'  {split}: {len(os.listdir(cfg[split]))} images')


In [ ]:
# Resume training from last.pt
import threading, time, shutil, os
from ultralytics import YOLO

DRIVE_BACKUP = '/content/drive/MyDrive/anti_uav_checkpoints_run2_resumed'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
stop_backup = threading.Event()

def backup_to_drive():
    while not stop_backup.is_set():
        time.sleep(600)
        # Find weights dir dynamically
        for root, dirs, files in os.walk('/content/runs'):
            if 'weights' in dirs:
                weights_dir = os.path.join(root, 'weights')
                for f in os.listdir(weights_dir):
                    src = os.path.join(weights_dir, f)
                    dst = os.path.join(DRIVE_BACKUP, f)
                    try:
                        shutil.copy2(src, dst)
                        print(f'[backup] {f} → Drive')
                    except Exception as e:
                        print(f'[backup] Failed: {e}')
                break

backup_thread = threading.Thread(target=backup_to_drive, daemon=True)
backup_thread.start()
print('Backup thread started')

# Copy last.pt locally for faster access
local_last = '/content/last.pt'
shutil.copy2(last_pt, local_last)
print(f'Copied last.pt to {local_last}')

model = YOLO(local_last)
results = model.train(
    data=data_yaml,
    resume=True,
    imgsz=640,
    batch=32,
    epochs=100,
    patience=30,
    fraction=1.0,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    save_period=10,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='anti_uav_run2_yolo26s_full',
)
stop_backup.set()
print(f'Training complete: {results.save_dir}')


In [ ]:
# Archive and save to Drive
import zipfile, os, shutil
runs_dir = results.save_dir
archive_drive = '/content/drive/MyDrive/anti_uav_run2_yolo26s_full.zip'

print('Archiving...')
with zipfile.ZipFile(archive_drive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/content')
            zf.write(filepath, arcname)
            print(f'  {arcname} ({os.path.getsize(filepath)/1e6:.1f} MB)')

print(f'Saved: {archive_drive} ({os.path.getsize(archive_drive)/1e6:.1f} MB)')
